# Data Preparation — Do Words Move Markets?

This notebook downloads 4 raw data sources and produces cleaned CSVs that feed the main analysis notebook (`02_full_analysis.ipynb`).

**Inputs:**

- Presidential speeches (Kaggle: `kboghe/presidentialspeeches`)
- Trump tweets (Kaggle: `zusmani/trumps-legacy`)
- Elon Musk tweets (Kaggle: `aryansingh0909/elon-musk-tweets-updated-daily`)
- SPY ETF daily prices (Yahoo Finance via `yfinance`)

**Outputs** (saved to `$ECLIPSE_DATA_DIR` or `./data/cleaned/`):

- `speech_clean.csv` — 995 rows
- `trump_clean.csv` — 56,571 rows
- `musk_clean.csv` — 24,450 rows
- `stocks_clean.csv` — 3,521 rows
- `all_text_clean.csv` — 82,016 rows
- `merged_cleaned_df.csv` — 59,627 rows (primary analysis dataset)

**Prerequisites:**

- Install dependencies from `requirements.txt` (includes `pandas`, `yfinance`, `kagglehub`, `openpyxl`).
- Set the `KAGGLE_USERNAME` and `KAGGLE_KEY` environment variables so `kagglehub` can authenticate.
- Optionally set `ECLIPSE_DATA_DIR` to control where cleaned CSVs are written; otherwise they land in `./data/cleaned/`.


## Setup

All cleaned outputs are written to a single configurable directory so the notebook is portable across local, Colab, and CI environments — no hard-coded user paths.


In [1]:
import os

os.environ["ECLIPSE_DATA_DIR"] = "../data/cleaned"

In [2]:
import kagglehub
import pandas as pd
import yfinance as yf

DATA_DIR = os.environ.get("ECLIPSE_DATA_DIR", "./data/cleaned")
os.makedirs(DATA_DIR, exist_ok=True)

print("Cleaned CSVs will be written to:", os.path.abspath(DATA_DIR))


Cleaned CSVs will be written to: /Users/oliviacai/Documents/do-words-move-markets/data/cleaned


## Text-source loader

The three text sources (speeches, Trump tweets, Musk tweets) follow the same pattern: download from Kaggle, rename a couple of columns, project to `(date, user_name, text)`, sort by date. We can then factor that pattern into a single helper and call it three times.


In [3]:
def load_and_clean_text(
    kaggle_id,
    file_name,
    rename_map,
    speaker_label=None,
    reader=pd.read_csv,
    sort_ascending=True,
    date_as_date_object=False,
):
    """Download a Kaggle text dataset and return a 3-column DataFrame (date, user_name, text).

    Parameters
    ----------
    kaggle_id : str
        Kaggle dataset slug, e.g. ``'kboghe/presidentialspeeches'``.
    file_name : str
        File inside the downloaded dataset folder to load.
    rename_map : dict
        Mapping from source column names to the standard ``date``, ``user_name``,
        and ``text`` names. Pass ``{}`` if the source already uses those names.
    speaker_label : str, optional
        Constant value to assign to ``user_name`` when the source lacks a
        username column (e.g. the Trump tweet dump).
    reader : callable
        Pandas reader function (``pd.read_csv`` or ``pd.read_excel``).
    sort_ascending : bool
        Sort direction for the final DataFrame.
    date_as_date_object : bool
        If True, cast the date column to ``datetime.date`` after sorting.
        Preserves the dtype the original notebook used for Musk's tweets.
    """
    raw_path = kagglehub.dataset_download(kaggle_id)
    df = reader(os.path.join(raw_path, file_name))
    df = df.rename(columns=rename_map)

    if speaker_label is not None:
        df["user_name"] = speaker_label

    df["date"] = pd.to_datetime(df["date"])
    df = df[["date", "user_name", "text"]]
    df = df.sort_values(by="date", ascending=sort_ascending).reset_index(drop=True)

    if date_as_date_object:
        df["date"] = df["date"].dt.date

    return df


### Speeches

Presidential speeches scraped from the Miller Center ([Kaggle source](https://www.kaggle.com/datasets/kboghe/presidentialspeeches)). We project to `(date, user_name, text)` where `user_name` here carries the president's name, so this source shares the same schema as the tweet sources for downstream concatenation.


In [4]:
speech = load_and_clean_text(
    kaggle_id="kboghe/presidentialspeeches",
    file_name="1presidential_speeches_with_metadata.xlsx",
    rename_map={"President": "user_name", "speech": "text"},
    reader=pd.read_excel,
    sort_ascending=False,  # original notebook sorted speeches newest-first
)

speech_path = os.path.join(DATA_DIR, "speech_clean.csv")
speech.to_csv(speech_path, index=False)
print(f"Saved {len(speech):,} rows to {speech_path}")
speech.head()


Saved 995 rows to ../data/cleaned/speech_clean.csv


,date,user_name,text
0,2020-01-08,Donald Trump,As long as I am President of the United States...
1,2020-01-03,Donald Trump,"Hello, everybody. Well, thank you very much. ..."
2,2019-10-27,Donald Trump,"Last night, the United States brought the worl..."
3,2019-09-25,Donald Trump,PRESIDENT TRUMP: Thank you very much. Thank...
4,2019-09-24,Donald Trump,PRESIDENT TRUMP: Thank you very much. Mr. ...


### Trump tweets

Trump's tweet archive ([Kaggle source](https://www.kaggle.com/datasets/zusmani/trumps-legacy)). The source dump lacks a username column, so we inject the literal `"realDonaldTrump"` label to keep the schema consistent with the other text sources.


In [5]:
trump = load_and_clean_text(
    kaggle_id="zusmani/trumps-legacy",
    file_name="Trumps Legcy.csv",
    rename_map={},
    speaker_label="realDonaldTrump",
)

trump_path = os.path.join(DATA_DIR, "trump_clean.csv")
trump.to_csv(trump_path, index=False)
print(f"Saved {len(trump):,} rows to {trump_path}")
trump.head()


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3.72M/3.72M [00:00<00:00, 9.49MB/s]

Extracting files...


Saved 56,571 rows to ../data/cleaned/trump_clean.csv


,date,user_name,text
0,2009-05-04 18:54:00,realDonaldTrump,Be sure to tune in and watch Donald Trump on L...
1,2009-05-05 01:00:00,realDonaldTrump,Donald Trump will be appearing on The View tom...
2,2009-05-08 13:38:00,realDonaldTrump,Donald Trump reads Top Ten Financial Tips on L...
3,2009-05-08 20:40:00,realDonaldTrump,New Blog Post: Celebrity Apprentice Finale and...
4,2009-05-12 14:07:00,realDonaldTrump,"""""""My persona will never be that of a wallflow..."


### Elon Musk tweets

Elon Musk's tweet archive ([Kaggle source](https://www.kaggle.com/datasets/aryansingh0909/elon-musk-tweets-updated-daily)). Source uses `Datetime`/`Username`/`Text` column names, which we remap to the shared `(date, user_name, text)` schema.


In [6]:
musk = load_and_clean_text(
    kaggle_id="aryansingh0909/elon-musk-tweets-updated-daily",
    file_name="elonmusk.csv",
    rename_map={"Datetime": "date", "Username": "user_name", "Text": "text"},
    date_as_date_object=True,  # match the original notebook's date dtype
)

musk_path = os.path.join(DATA_DIR, "musk_clean.csv")
musk.to_csv(musk_path, index=False)
print(f"Saved {len(musk):,} rows to {musk_path}")
musk.head()


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1.25M/1.25M [00:00<00:00, 3.46MB/s]

Extracting files...
Saved 24,450 rows to ../data/cleaned/musk_clean.csv


,date,user_name,text
0,2010-06-04,elonmusk,"Please ignore prior tweets, as that was someon..."
1,2011-12-01,elonmusk,Went to Iceland on Sat to ride bumper cars on ...
2,2011-12-01,elonmusk,I made the volume on the Model S http://t.co/w...
3,2011-12-03,elonmusk,"Great Voltaire quote, arguably better than Twa..."
4,2011-12-03,elonmusk,That was a total non sequitur btw


### Stocks (SPY)

SPY tracks the S&P 500 and gives a single, broad-market reference series; daily percent change provides the return signal we'll align with text events.


In [7]:
stocks = yf.download("SPY", start="2010-01-01", end="2024-01-01")

# yfinance returns a MultiIndex on columns when downloading a single ticker;
# flatten to the price field name so downstream code stays simple.
stocks.columns = [col[0] for col in stocks.columns]
stocks = stocks.reset_index().rename(columns={"Date": "date"})
stocks = stocks[["date", "Close"]]
stocks["return"] = stocks["Close"].pct_change()
stocks = stocks.dropna()

stocks_path = os.path.join(DATA_DIR, "stocks_clean.csv")
stocks.to_csv(stocks_path, index=False)
print(f"Saved {len(stocks):,} rows to {stocks_path}")
stocks.head()


[*********************100%***********************]  1 of 1 completed

Saved 3,521 rows to ../data/cleaned/stocks_clean.csv


,date,Close,return
1,2010-01-05,85.020851,0.002647
2,2010-01-06,85.080704,0.000704
3,2010-01-07,85.439850,0.004221
4,2010-01-08,85.724174,0.003328
5,2010-01-11,85.843910,0.001397


### Concatenated text

Stacking the three text sources into one long DataFrame lets the analysis notebook treat them uniformly and join against the daily stock series in a single pass; we rename `user_name` to `by` to match the column naming used downstream.


In [8]:
text = pd.concat([speech, trump, musk], ignore_index=True)
text["date"] = pd.to_datetime(text["date"])
text = text.rename(columns={"user_name": "by"})
text = text[["date", "text", "by"]]
text = text.sort_values(by="date").reset_index(drop=True)

text_path = os.path.join(DATA_DIR, "all_text_clean.csv")
text.to_csv(text_path, index=False)
print(f"Saved {len(text):,} rows to {text_path}")
text.head()


Saved 82,016 rows to ../data/cleaned/all_text_clean.csv


,date,text,by
0,1789-04-30,Fellow Citizens of the Senate and the House of...,George Washington
1,1789-10-03,Whereas it is the duty of all Nations to ackno...,George Washington
2,1790-01-08,Fellow Citizens of the Senate and House of Rep...,George Washington
3,1790-12-08,Fellow citizens of the Senate and House of Re...,George Washington
4,1790-12-29,"I the President of the United States, by my o...",George Washington


### Merged analysis dataset

An inner join on `date` drops weekends, holidays, and any text dated before SPY's history begins. This leaves only the rows where we have both a message and a market move to study. This is the primary input to `02_full_analysis.ipynb`.


In [9]:
df_merged = pd.merge(text, stocks, on="date", how="inner")

merged_path = os.path.join(DATA_DIR, "merged_cleaned_df.csv")
df_merged.to_csv(merged_path, index=False)
print(f"Saved {len(df_merged):,} rows to {merged_path}")
df_merged.info()


Saved 16,821 rows to ../data/cleaned/merged_cleaned_df.csv
<class 'pandas.DataFrame'>
RangeIndex: 16821 entries, 0 to 16820
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    16821 non-null  datetime64[us]
 1   text    16821 non-null  str           
 2   by      16821 non-null  str           
 3   Close   16821 non-null  float64       
 4   return  16821 non-null  float64       
dtypes: datetime64[us](1), float64(2), str(2)
memory usage: 657.2 KB
